# PHYS 142/242 — Assignment 2: Tunneling in the Double Potential Well

**Name:**  
**TSN:**  
**My $\alpha$:** $\alpha = 0.3 + d/5000$ where $d$ = last three digits of TSN → $\alpha =$  

Parts A–D only. Optional bonus E is skipped.

### How to use this notebook in VS Code
- **Pick the kernel first:** **Select Kernel** (top right) → *Python Environments* → your `.venv`.
- **Run a cell:** `Shift+Enter` runs it and moves on. `Ctrl+Enter` runs it and stays.
- **Add a cell:** hover between cells → **+ Code** / **+ Markdown**.
- **Switch cell type:** `Esc`, then `M` (Markdown) or `Y` (code).
- **Run top to bottom.** If things act strangely: **Restart**, then **Run All**.
- **Order tip:** the code runs A → B → C → D, but A(ii) compares with D's curve. Write the A(ii) answer after you've finished D.

## 0. Environment check
Run once. Every line should print a version number, not `MISSING`.

In [ ]:
import sys, importlib
print("Python", sys.version.split()[0], "at", sys.executable)
for pkg in ["numpy", "scipy", "matplotlib", "numba"]:
    try:
        print(f"{pkg:<11}", importlib.import_module(pkg).__version__)
    except ImportError:
        print(f"{pkg:<11} MISSING")

## Setup

**Numerical conventions (from the assignment):**
- Grid: $N_D+1 = 601$ points from $x_0=-4$ to $x_{N_D}=4$, $N_D=600$, $\Delta x = 8/600$. Only B(ii) changes the grid.
- Time step: $\epsilon = \Delta t = \pi/128$.
- Integrals: $\int f(x)\,dx \approx \Delta x \sum_{i=0}^{N_D} f(x_i)$, **including both endpoints**.
- $t_\text{tunnel}$: time of the first **major** minimum of $\langle x\rangle(t)$ (not a zero crossing, not a small wiggle).
- Units: $m=\hbar=1$.

**Potential and harmonic approximation:**
$$V(x)=\alpha x^4-2x^2+\frac1\alpha,\qquad x_\text{min}=\frac{1}{\sqrt\alpha},\qquad V_\pm(x)=4(x\mp x_\text{min})^2,\qquad \omega=2\sqrt2$$
$$\psi_\pm(x)=\left(\frac{\omega}{\pi}\right)^{1/4}\exp\!\left[-\frac{\omega}{2}(x\mp x_\text{min})^2\right],\qquad
\psi_S=\frac{\psi_++\psi_-}{\sqrt2},\quad \psi_A=\frac{\psi_+-\psi_-}{\sqrt2}$$

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse
import scipy.sparse.linalg
from numba import njit

In [ ]:
# Constants: TSN, ALPHA, grid (ND, x, DELTAX), time step (DELTAT), OMEGA, XMIN

In [ ]:
# Shared functions: V(x, alpha), psi_plus / psi_minus, ...

---
## Problem A [10 pts]: Path-integral tunneling

Start in the right well, $\psi(x,0)=\psi_+(x)$. Propagate one step at a time with the **midpoint** short-time kernel
$$\tilde K(x_b,\epsilon;x_a,0)=\exp\left[i\left(\frac12\frac{(x_b-x_a)^2}{\epsilon}-V\!\left(\frac{x_a+x_b}{2}\right)\epsilon\right)\right]$$
and normalize after **every** step: $\psi \leftarrow \psi/\sqrt{\int|\psi|^2dx}$.

For `results.json`: run to **at least $16\pi$** (2048 steps), longer if needed to see a full oscillation. Save $\langle x\rangle$ at $n = 0, 64, 128, \dots, 2048$.

### A(i) Run and plot
Follow one full slow oscillation (right → left → back). Plot $\langle x\rangle(t)$ and **mark the measured $t_\text{tunnel}$**. You may use $2\pi/\Delta E$ from B to choose the run length, but do **not** substitute $\pi/\Delta E$ for the measured time.

In [ ]:
# A(i): build kernel, evolve psi_+, compute <x>(t), find t_tunnel, plot

In [ ]:
# A(i): plot <x>(t) with t_tunnel marked

### A(ii) Read your result
Give the time and value of your chosen major minimum. Mark a faster fluctuation if one is resolved and distinguish it from the slow transfer. Compare with D's two-state curve: which feature of your results supports your account of any departures from a single sinusoid?

**Answer (A(ii)):** *(write this after finishing D)*

---
## Problem B [10 pts]: Numerical eigenstates

Find $E_0, E_1$ and $\phi_0,\phi_1$ of $\hat H=-\frac12\partial_x^2+V(x)$ with the $(N_D+1)\times(N_D+1)$ three-point matrix
$$H_{ii}=\frac{1}{\Delta x^2}+V(x_i),\qquad H_{i,i+1}=H_{i+1,i}=-\frac{1}{2\Delta x^2}.$$
- **Endpoints:** keep all $N_D+1$ points. Just drop the outside neighbor in the first and last rows.
- **Solver:** `scipy.sparse.linalg.eigsh(H, k=2, sigma=0, which="LM")` (shift-invert), as in the Lecture 11 demo.

In [ ]:
# B: function that builds the sparse Hamiltonian and returns E0, E1, phi0, phi1 for a given (alpha, L, ND)

### B(i) Validate first
Compute $E_0,E_1$ at $\alpha=0.4$ and $0.6$ and compare with Banerjee & Bhatnagar Table I. Their $\lambda=\alpha/4$, so use $\lambda=0.10,\,0.15$:

| $\lambda$ | $\alpha$ | $\epsilon_0$ | $\epsilon_1$ |
|---|---|---|---|
| 0.10 | 0.4 | 1.234 507 162 786 0 | 1.346 940 868 922 5 |
| 0.15 | 0.6 | 1.062 499 247 956 5 | 1.421 086 890 539 3 |

Report how many digits agree. Use the comparison and your convergence checks to discuss what may limit the agreement.

In [ ]:
# B(i): compare with the table

**Answer (B(i)):**

### B(ii) Check convergence at your α
Report $E_0, E_1, \Delta E$ on the original grid, then test:

| Grid | Interval | $N_D$ | $\Delta x$ |
|---|---|---|---|
| Original | $[-4,4]$ | 600 | 8/600 |
| Half spacing | $[-4,4]$ | 1200 | 8/1200 |
| Larger box | $[-6,6]$ | 900 | 8/600 |

Tabulate each gap and its change from the original. Assess sensitivity to grid spacing and box size, and justify the digits you quote. **Identify the code that sets the grid spacing and updates the Hamiltonian.** Use the original-grid results from here on.

In [ ]:
# B(ii): convergence table

**Answer (B(ii)):**

### B(iii) Plot eigenfunctions
Normalize with $\Delta x\sum_i|\phi_n(x_i)|^2=1$. Choose $\phi_0$ positive and the **right-hand lobe** of $\phi_1$ positive. Plot both together, with $V(x)$ in a **separate panel sharing the x-axis**.

In [ ]:
# B(iii): normalize, fix signs, two-panel plot

**Answer (B(iii)):**

---
## Problem C [10 pts]: Harmonic-approximation energies

$$E_S=\frac{\int\psi_S^*\hat H\psi_S\,dx}{\int|\psi_S|^2dx},\qquad E_A=\frac{\int\psi_A^*\hat H\psi_A\,dx}{\int|\psi_A|^2dx}$$
Use **B's Hamiltonian matrix** and grid sums. **Keep the denominators**: $\psi_S,\psi_A$ are slightly unnormalized.

### C(i) Compare energies
Compare $E_S, E_A, E_A-E_S$ with $E_0, E_1, \Delta E$. Report relative errors (e.g. $|E_S-E_0|/|E_0|$). Use the signed errors $E_S-E_0$ and $E_A-E_1$ to account for the gap error. Which is more accurate in relative terms: the energies or the gap?

In [ ]:
# C(i): E_S, E_A and their errors

**Answer (C(i)):**

### C(ii) Compare wave functions
Normalize $\psi_S$, then overlay $|\psi_S|$ and $|\phi_0|$ on a **logarithmic y-axis**. Mark a discrepancy in the barrier region and relate it to the limits of the local harmonic approximation. Use the energy errors from (i) to show why the gap's relative error can be large even when the energies' relative errors are small.

In [ ]:
# C(ii): log-scale overlay

**Answer (C(ii)):**

---
## Problem D [10 pts]: Compare two evolution methods

Initial state for **both** methods: $\psi(x,0)=(\phi_0+\phi_1)/\sqrt2$, with B's signs. Set $T=2\pi/\Delta E$ and evolve to the **first time step at or after $T$**.

### D(i) From the eigenstates
$$\psi(x,t)=\frac{1}{\sqrt2}\left(e^{-iE_0t/\hbar}\phi_0(x)+e^{-iE_1t/\hbar}\phi_1(x)\right)$$
Show that $\langle x\rangle(t)=x_{01}\cos(\Delta E\,t/\hbar)$ with $x_{01}=\int\phi_0\,x\,\phi_1\,dx$, and hence $t_\text{tunnel}=\pi\hbar/\Delta E$. Identify where normalization and symmetry enter your derivation.

**Derivation (D(i)):**

In [ ]:
# D(i): <x>_eig(t) on the time grid, plus x_mean_eig at the 33 export times (n = 0, 64, ..., 2048)

### D(ii) With the path integral
Reuse A's code with the two-state initial wave function above.

In [ ]:
# D(ii): path-integral evolution of (phi0 + phi1)/sqrt(2)

### D(iii) Compare
- Overlay both $\langle x\rangle(t)$ curves and report $\max_{0\le n\epsilon\le T}\left|\langle x\rangle_\text{PI}(n\epsilon)-\langle x\rangle_\text{eig}(n\epsilon)\right|$.
- Overlay both probability densities at the steps nearest to $0,\ t_\text{tunnel},\ T$. Label the methods and the actual times.
- At the maximum discrepancy, report the time and both means.
- Identify a discrepancy pattern and a possible numerical cause. Propose one targeted check and predict its outcome (no extra run needed).
- *Optional:* animate both densities (.gif/.mp4). No extra points.

In [ ]:
# D(iii): overlay <x> curves, max discrepancy, density snapshots

**Answer (D(iii)):**

---
## Autograded results: `results.json`
Run this **last**, once every variable exists. Upload `results.json` together with the source files to "Assignment 2 Code".

| Field | Contents |
|---|---|
| `tsn`, `alpha` | your TSN (string) and $\alpha$ |
| `x_mean_pi` | A: $\langle x\rangle$ at $n=0,64,\dots,2048$ (33 values), starting from $\psi_+$ |
| `t_tunnel` | A: measured $t_\text{tunnel}$ |
| `E0`, `E1` | B: original grid |
| `phi0`, `phi1` | B: 601 values each, normalized, B's signs |
| `E_S`, `E_A` | C |
| `x_mean_eig` | D(i): at the same 33 times |

In [ ]:
# Template from the assignment (the Problem E lines are removed because E is skipped)
results = {
    "tsn": TSN,
    "alpha": ALPHA,
    "x_mean_pi": x_mean_pi.tolist(),
    "t_tunnel": float(t_tunnel),
    "E0": float(E0), "E1": float(E1),
    "phi0": phi0.tolist(), "phi1": phi1.tolist(),
    "E_S": float(E_S), "E_A": float(E_A),
    "x_mean_eig": x_mean_eig.tolist(),
}
with open("results.json", "w") as f:
    json.dump(results, f)
print("wrote results.json")

---
## AI use appendix
*(Required by the syllabus.)* List the tools you used and what for, your prompts (or a conversation link), and what the tools got wrong and how you caught it.